# Example 01: Basic Chat（基础对话）

单轮问答 + 多轮对话上下文管理

**Demonstrates:**
- Single-turn Q&A
- Multi-turn conversation with history
- Accessing response fields (`content`, `usage`, `finish_reason`)

**Prerequisites:**
```bash
pip install openai
# Start the Hy3 server first (see quickstart.md)
```

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    base_url=os.environ.get("HY3_BASE_URL", "http://127.0.0.1:8000/v1"),
    api_key=os.environ.get("HY3_API_KEY", "EMPTY"),
)

MODEL = os.environ.get("HY3_MODEL", "hy3")

## Single-turn（单轮问答）

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "What is the capital of France? Answer in one sentence."}
    ],
    temperature=0.9,
    top_p=1.0,
    # reasoning_effort: "no_think" (default) = direct answer, no chain-of-thought
    extra_body={"chat_template_kwargs": {"reasoning_effort": "no_think"}},
)

msg = response.choices[0].message
usage = response.usage

print(f"Response     : {msg.content}")
print(f"Finish reason: {response.choices[0].finish_reason}")
print(f"Tokens — prompt: {usage.prompt_tokens}, completion: {usage.completion_tokens}, total: {usage.total_tokens}")

**Sample output:**
```
Response     : The capital of France is Paris.
Finish reason: stop
Tokens — prompt: 20, completion: 9, total: 29
```

## Multi-turn（多轮对话）

每轮结束后将 assistant 回复追加到 `messages`，下一轮自动携带完整上下文。

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful geography tutor. Keep answers concise."},
]

turns = [
    "What is the highest mountain in the world?",
    "How tall is it in feet?",
    "When was it first summited?",
]

for user_input in turns:
    messages.append({"role": "user", "content": user_input})
    print(f"User: {user_input}")

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.9,
        top_p=1.0,
        extra_body={"chat_template_kwargs": {"reasoning_effort": "no_think"}},
    )

    assistant_reply = response.choices[0].message.content
    print(f"Hy3 : {assistant_reply}\n")

    # Append to keep context for the next turn
    messages.append({"role": "assistant", "content": assistant_reply})

**Sample output:**
```
User: What is the highest mountain in the world?
Hy3 : Mount Everest, located in the Himalayas on the Nepal-Tibet border.

User: How tall is it in feet?
Hy3 : Mount Everest stands at 29,032 feet (8,849 meters) above sea level.

User: When was it first summited?
Hy3 : It was first summited on May 29, 1953, by Sir Edmund Hillary and Tenzing Norgay.
```